RAG application using Typesense

In [5]:
import typesense
import os
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
client = typesense.Client({
    'nodes': [{
        'host': os.getenv("TYPESENSE_HOST"), # For Typesense Cloud use xxx.a1.typesense.net
        'port': '443', # For Typesense Cloud use 443
        'protocol': 'https' # For Typesense Cloud use https
    }],
    'api_key': os.getenv("TYPESENSE_API_KEY"),
    'connection_timeout_seconds': 2
})

In [7]:
books_schema = {
    'name': 'books',
    'fields': [
        {'name': 'title', 'type': 'string' },
        {'name': 'authors', 'type': 'string[]', 'facet': True },
        {'name': 'publication_year', 'type': 'int32', 'facet': True },
        {'name': 'ratings_count', 'type': 'int32' },
        {'name': 'average_rating', 'type': 'float' }
    ],
    'default_sorting_field': 'ratings_count'
}
print(client.collections.create(books_schema))

{'created_at': 1779224570, 'curation_sets': [], 'default_sorting_field': 'ratings_count', 'enable_nested_fields': False, 'fields': [{'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'title', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'authors', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string[]'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'publication_year', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'ratings_count', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': T

In [8]:
with open('books.jsonl', 'r', encoding='utf-8') as json_file:
    data = json_file.read()
    client.collections['books'].documents.import_(data)

In [13]:
search_parameters = {
    'q': 'harry potter',
    'query_by': 'title,authors',
    'filter_by': 'publication_year:>2000 && average_rating:>4.0',
    'sort_by': 'ratings_count:desc',
    'facet_by': 'publication_year',
    'max_facet_values': 5
}
client.collections['books'].documents.search(search_parameters)
# search_results = client.collections['books'].documents.search(search_parameters)
# print(search_results)

{'facet_counts': [{'counts': [{'count': 2,
     'highlighted': '2005',
     'value': '2005'},
    {'count': 2, 'highlighted': '2003', 'value': '2003'},
    {'count': 1, 'highlighted': '2011', 'value': '2011'},
    {'count': 1, 'highlighted': '2010', 'value': '2010'},
    {'count': 1, 'highlighted': '2008', 'value': '2008'}],
   'field_name': 'publication_year',
   'sampled': False,
   'stats': {'avg': 2006.2222222222222,
    'max': 2011.0,
    'min': 2003.0,
    'sum': 18056.0,
    'total_values': 7}}],
 'found': 9,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.61,
    'id': '25',
    'image_url': 'https://images.gr-assets.com/books/1474171184m/136251.jpg',
    'publication_year': 2007,
    'ratings_count': 1746574,
    'title': 'Harry Potter and the Deathly Hallows'},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': '<mark>Harry</mark> <mark>Potter</mark> and the Deathly Hallows'}},
   'highlights': [{'f

In [33]:
### Langchain + Typesense + Groq LLM + RAG Application

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import FakeEmbeddings
from langchain_groq import ChatGroq

import os
from dotenv import load_dotenv
load_dotenv()


True

In [34]:
loader = TextLoader("test.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

# FakeEmbeddings generates random vectors - fine for learning the RAG pipeline
# Replace with a real embeddings model (e.g. OpenAIEmbeddings) in production
embeddings = FakeEmbeddings(size=384)


In [35]:
docsearch = Typesense.from_documents(
  docs, 
  embeddings, 
  typesense_client_params={
    "host": os.getenv("TYPESENSE_HOST"), # For Typesense Cloud use xxx.a1.typesense.net
    "port": '443', # For Typesense Cloud use 443
    "protocol": 'https', # For Typesense Cloud use https
    "typesense_api_key": os.getenv("TYPESENSE_API_KEY"),
    "typesense_collection_name": "lang-chain"
  }
)

In [37]:
query = "What is Artificial Intelligence?"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

Artificial Intelligence: Transforming the Future of Humanity

Introduction

Artificial intelligence is transforming global society.
It combines computer science, mathematics, and domain expertise.
At its core, AI seeks to replicate or augment human cognitive abilities.
These abilities include learning, reasoning, perception, and decision-making.
Modern AI systems are powered by data, algorithms, and compute resources.
The recent progress in machine learning has accelerated innovation in many fields.

In healthcare, AI assists doctors with diagnosis and treatment planning.
Algorithms can analyze medical images and detect abnormalities earlier.
AI helps manage patient records and predict health outcomes.
Personalized medicine is becoming possible through intelligent data analysis.


In [38]:
### Retriever
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'FakeEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x0000015150906A50>, search_kwargs={})

In [39]:
query = "Artificial Intelligence indepth explanation?"
retriever.invoke(query)[0].page_content

'In education, AI supports adaptive learning platforms and tutoring systems.\nStudents receive instruction that matches their pace and style.\nTeachers can use AI insights to identify student strengths and gaps.\nThese tools can make education more accessible and effective worldwide.\n\nIn finance, AI improves fraud detection and automates financial services.\nIntelligent systems scan transactions and flag suspicious behavior.\nInvestment strategies are optimized with predictive analytics.\nCustomers use AI-powered chatbots for faster and more convenient support.\n\nIn transportation, AI contributes to safer and more efficient travel.\nSelf-driving vehicles navigate roads with sensors and intelligent planning.\nTraffic systems can adjust signals based on real-time conditions.\nLogistics networks are optimized for faster delivery and lower emissions.'